In [1]:
!pip install -q ultralytics opencv-python numpy torch
!pip install -q onnx==1.17.0 onnxslim onnxruntime onnxruntime-gpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.7/238.7 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.8/249.8 MB 7.4 MB/s eta 0:00:00


In [2]:
import os
import time
import numpy as np
import cv2
import torch
import yaml
from ultralytics import YOLO

# ============================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN OUTPUT (WEIGHTS & YAML CŨ)
# ============================================================
CURRENT_ACCOUNT = 2
BASE_INPUT_DIR = "/kaggle/input/datasets/vdt1501/acc2-trained-weight-mlops-sau-benh-cay-trong"

WEIGHTS_PATH = os.path.join(
    BASE_INPUT_DIR,
    f"runs/detect/Agricultural_YOLO26s_Project/Dual_T4_Account_{CURRENT_ACCOUNT}/weights/best.pt"
)

OLD_YAML_PATH = os.path.join(
    BASE_INPUT_DIR,
    f"split_metadata/data_account_{CURRENT_ACCOUNT}.yaml"
)

OLD_TEST_TXT_PATH = os.path.join(
    BASE_INPUT_DIR,
    "split_metadata/test.txt"
)

if not os.path.exists(WEIGHTS_PATH):
    raise FileNotFoundError(f"Không tìm thấy weights tại: {WEIGHTS_PATH}")

print(f"✅ Tải mô hình từ: {WEIGHTS_PATH}", flush=True)
model = YOLO(WEIGHTS_PATH)

# ============================================================
# 1.5. TỰ ĐỘNG TÌM VÀ SỬA LẠI ĐƯỜNG DẪN ẢNH (CHỐNG LỖI NO SUCH FILE)
# ============================================================
print("⏳ Đang quét để tìm dataset ảnh gốc trong /kaggle/input/...", flush=True)

actual_img_dir = None
# Kiểm tra các đường dẫn mount phổ biến của Kaggle
for p in [
    "/kaggle/input/datasets/vdt1501/unified-disease-leaf-ip102/unified_dataset/images",
    "/kaggle/input/unified-disease-leaf-ip102/unified_dataset/images",
    "/kaggle/input/unified-disease-leaf-ip102/images"
]:
    if os.path.exists(p) and len(os.listdir(p)) > 0:
        actual_img_dir = p
        break

# Nếu không nằm trong path dự đoán, quét tự động
if not actual_img_dir:
    for root, dirs, files in os.walk("/kaggle/input"):
        if any(f.endswith(('.jpg', '.jpeg', '.png')) for f in files):
            actual_img_dir = root
            break

if not actual_img_dir:
    raise FileNotFoundError(
        "❌ KHÔNG TÌM THẤY dataset ảnh gốc!\n"
        "👉 BẠN BẮT BUỘC PHẢI:\n"
        "1. Nhấn 'Add Input' ở tab bên phải.\n"
        "2. Tìm và thêm dataset 'unified-disease-leaf-ip102' vào notebook này."
    )

print(f"✅ Tìm thấy thư mục ảnh thực tế tại: {actual_img_dir}", flush=True)

# Đọc test.txt cũ và ghi đè đường dẫn mới vào file tạm
NEW_TEST_TXT_PATH = "/kaggle/working/fixed_test.txt"
with open(OLD_TEST_TXT_PATH, "r") as f_in, open(NEW_TEST_TXT_PATH, "w") as f_out:
    for line in f_in:
        filename = os.path.basename(line.strip())
        if filename:
            f_out.write(os.path.join(actual_img_dir, filename) + "\n")

# ============================================================
# 1.6. SỬA LẠI FILE YAML
# ============================================================
with open(OLD_YAML_PATH, "r", encoding="utf-8") as f:
    yaml_data = yaml.safe_load(f)

# Ghi đè đường dẫn test bằng file mới vừa tạo
yaml_data["test"] = NEW_TEST_TXT_PATH

# Xóa key 'path' nếu có, để ngăn YOLO tự ý scan thư mục gốc gây lỗi cache
if "path" in yaml_data:
    del yaml_data["path"]

FIXED_YAML_PATH = "/kaggle/working/fixed_data_account_1.yaml"
with open(FIXED_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(yaml_data, f, sort_keys=False, allow_unicode=True)

print(f"✅ Đã tạo file YAML và test.txt hợp lệ.", flush=True)

# ============================================================
# 2. ĐÁNH GIÁ MÔ HÌNH (EVALUATION)
# ============================================================
print("⏳ Đang chạy đánh giá trên tập Test...", flush=True)
val_results = model.val(data=FIXED_YAML_PATH, split='test', device="0,1", batch=86, plots=False)
metrics_dict = val_results.results_dict
print(f"✅ mAP50-95: {metrics_dict.get('metrics/mAP50-95(B)', 0.0):.4f}", flush=True)

# ============================================================
# 3. TÍNH TOÁN PSI DATA DRIFT
# ============================================================
print("⏳ Đang tính toán PSI Data Drift...", flush=True)

with open(NEW_TEST_TXT_PATH, "r", encoding="utf-8") as f:
    test_img_paths = [line.strip() for line in f.readlines() if line.strip()]

brightness_values = []
for p in test_img_paths[:150]:
    if os.path.exists(p):
        img = cv2.imread(p)
        if img is not None:
            v_channel = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)[:, :, 2]
            brightness_values.append(np.mean(v_channel))

brightness_values = np.array(brightness_values) if len(brightness_values) > 0 else np.array([128.0])

h_val, _ = np.histogram(brightness_values, bins=10, range=(0, 255))
h_val = h_val.astype(np.float64)
h_val = h_val / max(h_val.sum(), 1)
h_train = np.ones(10, dtype=np.float64) / 10

h_val = np.clip(h_val, 1e-4, None)
h_train = np.clip(h_train, 1e-4, None)
psi_value = np.sum((h_val - h_train) * np.log(h_val / h_train))

# ============================================================
# 4. BENCHMARK LATENCY
# ============================================================
print("⏳ Đang Benchmark Latency GPU...", flush=True)
device = torch.device("cuda:0")
model.model.to(device).half()
model.model.eval()

dummy_input = torch.randn(1, 3, 640, 640, device=device, dtype=torch.float16)

with torch.no_grad():
    for _ in range(20):
        _ = model.model(dummy_input)
torch.cuda.synchronize()

edge_latencies = []
with torch.no_grad():
    for _ in range(50):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = model.model(dummy_input)
        torch.cuda.synchronize()
        edge_latencies.append((time.perf_counter() - t0) * 1000)

p50_latency = np.percentile(edge_latencies, 50)
p95_latency = np.percentile(edge_latencies, 95)
p99_latency = np.percentile(edge_latencies, 99)

import shutil

# ============================================================
# 5. XUẤT BÁO CÁO & EXPORT ONNX (ĐÃ SỬA LỖI READ-ONLY)
# ============================================================
print("\n==================================================", flush=True)
print("📊 BÁO CÁO MLOPS PIPELINE (STANDALONE RUN):", flush=True)
print(f"   - mAP@0.5:0.95 : {metrics_dict.get('metrics/mAP50-95(B)', 0.0):.4f}", flush=True)
print(f"   - PSI Drift    : {psi_value:.4f} -> {'🔴 ALERT DRIFT!' if psi_value > 0.2 else '🟢 STABLE'}", flush=True)
print(f"   - p50 Latency  : {p50_latency:.2f} ms", flush=True)
print(f"   - p95 Latency  : {p95_latency:.2f} ms", flush=True)
print(f"   - p99 Latency  : {p99_latency:.2f} ms", flush=True)
print("==================================================", flush=True)

# CƠ CHẾ SỬA LỖI: Copy weights sang thư mục /kaggle/working/ (có quyền ghi)
print("📦 Đang chuẩn bị môi trường để export ONNX...", flush=True)
pt_writable_path = "/kaggle/working/best.pt"
shutil.copy(WEIGHTS_PATH, pt_writable_path)

# Load lại model từ đường dẫn mới
model_for_export = YOLO(pt_writable_path)

print("📦 Đang nén mô hình sang ONNX...", flush=True)
try:
    # Export từ model mới, file .onnx sẽ tự động lưu vào /kaggle/working/
    # Thêm device="0" để ép export chạy trên GPU cho nhanh
    onnx_path = model_for_export.export(format="onnx", half=True, simplify=True, opset=12, device="0")
    print(f"✅ Hoàn tất! File ONNX FP16 được lưu tại: {onnx_path}", flush=True)
except Exception as e_fp16:
    print(f"⚠️ Export FP16 gặp vấn đề: {str(e_fp16)[:80]}...", flush=True)
    print("🔄 Đang tự động chuyển sang chế độ an toàn (FP32)...", flush=True)
    onnx_path = model_for_export.export(format="onnx", half=False, simplify=True, opset=12, device="0")
    print(f"✅ Hoàn tất! File ONNX FP32 được lưu tại: {onnx_path}", flush=True)

print("\n🎉 TOÀN BỘ PIPELINE MLOps ĐÃ HOÀN THÀNH XUẤT SẮC!", flush=True)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Tải mô hình từ: /kaggle/input/datasets/vdt1501/acc2-trained-weight-mlops-sau-benh-cay-trong/runs/detect/Agricultural_YOLO26s_Project/Dual_T4_Account_2/weights/best.pt
⏳ Đang quét để tìm dataset ảnh gốc trong /kaggle/input/...
✅ Tìm thấy thư mục ảnh thực tế tại: /kaggle/input/datasets/vdt1501/unified-disease-leaf-ip102/unified_dataset/images
✅ Đã tạo file YAML và test.txt hợp lệ.
⏳ Đang chạy đánh giá trên tập Test...
Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
YOLO26s summary (fused): 122 layers, 9,555,351 parameters, 0 gradients, 21.3 GFLOPs
WARNING ⚠️ val: